<div dir="rtl">

# NB04-M01 — Translation Inventory

## هدف
ممیزی دو ترجمه فارسی Legacy از نظر هویت مترجم، Hash محلی، 6236 آیه، Alignment دقیق با قرآن Canonical و وضعیت حقوق استفاده.

این Notebook هیچ ترجمه‌ای را به‌خاطر حضور در Repository به‌صورت خودکار Active نمی‌کند.

</div>

In [ ]:
# NB04-C01 — Imports and project root
import json
import sys
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    (p for p in [current, *current.parents] if (p / ".git").exists() and (p / "data").exists()),
    None,
)
if project_root is None:
    raise RuntimeError("QRAG project root not found.")

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from qrag_phase1.foundation import collect_git_info, write_json_atomic
from qrag_phase1.quran import parse_quran_pipe_file, read_jsonl, sha256_file, validate_quran_records
from qrag_phase1.translation import build_translation_records, validate_translation_alignment

assert collect_git_info(project_root)["branch"] == "qrag-phase1"

<div dir="rtl">

# NB04-M02 — Upstream Quran Gate

Canonical Quran باید قبلاً توسط NB03 ساخته شده باشد.

</div>

In [ ]:
# NB04-C02 — Load canonical Quran and translation contract
contract = json.loads(
    (project_root / "config/phase1/translation_contract_v1.json").read_text(encoding="utf-8")
)
evidence = json.loads(
    (project_root / "data/phase1/source_evidence/translation_tanzil_evidence_v1.json").read_text(encoding="utf-8")
)
quran_path = project_root / contract["canonical_quran_input"]
assert quran_path.is_file(), "Run NB03 first."
quran_rows = read_jsonl(quran_path)
assert len(quran_rows) == 6236

<div dir="rtl">

# NB04-M03 — Parse and Validate Each Translation

هر فایل باید 6236 کلید سوره/آیه داشته باشد و ترتیبش با قرآن Canonical دقیقاً یکی باشد.

</div>

In [ ]:
# NB04-C03 — Translation inventory
quran_contract = json.loads(
    (project_root / "config/phase1/quran_canonical_contract_v1.json").read_text(encoding="utf-8")
)
inventory = []

for source in contract["sources"]:
    raw_path = project_root / source["raw_file"]
    assert raw_path.is_file(), f"Missing translation raw file: {raw_path}"

    parsed = parse_quran_pipe_file(raw_path)
    structure = validate_quran_records(
        parsed["records"],
        quran_contract["expected_verse_counts_by_surah"],
    )
    assert structure["status"] == "PASS"

    raw_hash = sha256_file(raw_path)
    translation_rows = build_translation_records(
        parsed["records"],
        source["translation_id"],
        source["source_id"],
        raw_hash,
    )
    alignment = validate_translation_alignment(
        quran_rows,
        translation_rows,
    )
    assert alignment["status"] == "PASS"

    inventory.append({
        "translation_id": source["translation_id"],
        "source_id": source["source_id"],
        "translator": source["translator"],
        "raw_file": source["raw_file"],
        "raw_sha256": raw_hash,
        "raw_size_bytes": raw_path.stat().st_size,
        "record_count": len(translation_rows),
        "alignment": alignment,
        "rights_status": source["rights_status"],
        "activation_policy": source["activation_policy"],
    })

print(json.dumps(inventory, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB04-M04 — Rights Gate

Tanzil قابلیت دانلود ترجمه‌ها را مستند کرده، اما مجوز متن قرآن عربی را به‌طور خودکار به ترجمه‌های hosted تعمیم نمی‌دهیم.

</div>

In [ ]:
# NB04-C04 — Rights and attribution gate
assert (
    evidence["rights_interpretation"]["status"]
    == "NOT_VERIFIED_FOR_PRODUCT_REPUBLICATION"
)

for item in inventory:
    assert item["rights_status"] == "evidence_required"
    assert item["activation_policy"] == "quarantine_until_rights_verified"

print("Translation rights gate: QUARANTINE")

<div dir="rtl">

# NB04-M05 — Inventory Acceptance

Structure و Alignment می‌توانند PASS شوند، درحالی‌که Product activation به‌علت حقوق منبع Blocked بماند.

</div>

In [ ]:
# NB04-C05 — Persist inventory acceptance
reports = project_root / "reports/phase1"
reports.mkdir(parents=True, exist_ok=True)

report = {
    "schema_version": "qrag_translation_inventory_v1",
    "translation_count": len(inventory),
    "translations": inventory,
    "structural_status": "PASS",
    "rights_status": "BLOCKED_PENDING_SOURCE_SPECIFIC_EVIDENCE",
}
report_path = reports / "translation_inventory_v1.json"
write_json_atomic(report_path, report)

acceptance = {
    "schema_version": "qrag_nb04_acceptance_v1",
    "objective_status": "ACHIEVED",
    "acceptance_status": "PASS_WITH_LIMITATIONS",
    "failed_checks": [],
    "warnings": [
        "TRANSLATION_PRODUCT_RIGHTS_NOT_VERIFIED",
    ],
    "structurally_valid_translation_count": len(inventory),
    "active_translation_count": 0,
    "next_notebook": "notebooks/phase1/05_translation_ingestion_and_acceptance.ipynb",
}
acceptance_path = reports / "translation_inventory_acceptance_v1.json"
write_json_atomic(acceptance_path, acceptance)

assert json.loads(report_path.read_text(encoding="utf-8")) == report
assert json.loads(acceptance_path.read_text(encoding="utf-8")) == acceptance
print(json.dumps(acceptance, ensure_ascii=False, indent=2))

<div dir="rtl">

# NB04-M06 — جمع‌بندی

اگر Run All واقعی موفق باشد، دو ترجمه از نظر ساختاری قابل استفاده پژوهشی هستند؛ اما تا Evidence حقوقی مستقل به Active Product Corpus منتقل نمی‌شوند.

</div>